# P4 · IDEA 1 — `M` как сэмплер соседей вместо `LastNeighborLoader`

**Идея.** Сейчас GNN-эмбеддинг (`MGraphAttentionEmbedding`) получает соседей из `LastNeighborLoader` — это **последние K событий** по времени (рекуррентность). Заменяем выбор соседей на **аффинность из плотной памяти** `M[u,:]`: для размеченного юзера кормим GNN его **top-K item'ов по `M`**, а не самые свежие.

**Это НЕ ансамбль.** `M` лишь **отбирает, какие соседние памяти увидит единственная голова** — это выбор входа ДО головы, структурно идентичный `LastNeighborLoader` (тоже отбирает K соседей). На инференсе никакого второго предсказателя: голова `node_pred(z)` одна. (Запрещённое — `logits = head(z) + M[u,:]`; здесь этого нет.)

**Этот ноутбук — pre-experiment проба (фаза P4). ОБУЧЕНИЯ НЕТ.** Один офлайн строго-каузальный проход подтверждает/опровергает предпосылку, прежде чем тратить P5-прогон:

> top-K по `M` **отличается** от top-K по рекуррентности **И** ловит больше позитивной массы метки дня (особенно у недообслуженных юзеров: тёплых, у кого recency обрезан на K, и холодных).

**Каузальность.** И `M`, и `LastNeighborLoader` пишутся ТОЛЬКО рёбрами `t < lab_ts0` (strict pm-split как в p4), читаются на `t_read = lab_ts0`. Паритет `M`-only test NDCG@10 ≈ 0.520 — якорь доверия перед любой метрикой ниже.

**Гейт go/no-go (заполняется ячейками ниже):** медиана Jaccard(recency, M) < ~0.6 **И** `m_ceil − rec_ceil` > ~+0.01 на тёплых-обрезанных/холодных когортах **И** восстановление позитивов > 0 **И** стоимость селекции пренебрежима.

In [1]:
# Setup — harness из p3/p4 (verbatim) + реальный LastNeighborLoader. Ядро свежее, поэтому копируем код.
import numpy as np, polars as pl, plotly.express as px, plotly.graph_objects as go, timeit, sys
import torch
from sklearn.metrics import ndcg_score
from tgb.nodeproppred.dataset_pyg import PyGNodePropPredDataset
from tgb.nodeproppred.evaluate import Evaluator
from torch_geometric.loader import TemporalDataLoader
REPO = "/Users/aleksandrpanysev/Documents/GitHub/2Q_2026_tgn_user_item"
if REPO not in sys.path: sys.path.insert(0, REPO)
from models.mtgn import LastNeighborLoader   # реальный recency-сэмплер (gold standard)

def load_dataset(name, bs=200):
    ds = PyGNodePropPredDataset(name=name, root="datasets")
    data = ds.get_TemporalData()
    tr, va, te = data.train_val_test_split(val_ratio=0.15, test_ratio=0.15)
    return dict(ds=ds, data=data, num_classes=ds.num_classes, num_nodes=data.num_nodes,
                evaluator=Evaluator(name=name), eval_metric=ds.eval_metric,
                loaders={s: TemporalDataLoader(d, batch_size=bs) for s, d in [("train", tr), ("val", va), ("test", te)]})

def make_rank_phi(DS):  # rank/ECDF по train (каузально)
    tr, _, _ = DS["data"].train_val_test_split(val_ratio=0.15, test_ratio=0.15)
    sw = np.sort(tr.msg[:, 0].numpy()); n = len(sw)
    return lambda w: np.searchsorted(sw, w, side="right") / n

class HawkesMemory:
    def __init__(self, N, C, alpha, phi):
        self.C = C; self.M = np.zeros((N, C)); self.tlast = np.zeros((N, C)); self.alpha = float(alpha); self.phi = phi
    def update(self, src, dst, t, w):
        if src.size == 0: return
        idx = src.astype(np.int64) * self.C + dst.astype(np.int64); tb = float(t.max())
        uniq, inv = np.unique(idx, return_inverse=True); add = np.bincount(inv, weights=self.phi(w), minlength=uniq.size)
        fM, fT = self.M.reshape(-1), self.tlast.reshape(-1)
        fM[uniq] = (fM[uniq] * np.exp(-self.alpha * np.clip(tb - fT[uniq], 0, None)) + add) if self.alpha > 0 else fM[uniq] + add
        fT[uniq] = tb
    def read(self, users, t_read):
        r = self.M[users]
        return r * np.exp(-self.alpha * np.clip(t_read - self.tlast[users], 0, None)) if self.alpha > 0 else r

DS = load_dataset("tgbn-genre"); DCHAR = 86400.0; KAPPA = 25
alpha = np.log(2) / (KAPPA * DCHAR); phi = make_rank_phi(DS)
K = 30  # = paper genre num_last_neighbours x → размер loader и бюджет M-top-k одинаковы (apples-to-apples)
print(f"genre: nodes={DS['num_nodes']} classes={DS['num_classes']} edges={DS['data'].src.numel()}  "
      f"alpha={alpha:.3e}  K={K}")

genre: nodes=1505 classes=513 edges=17858395  alpha=3.209e-07  K=30


In [2]:
# Precondition: item node-id == class-id ⇒ M-колонки = item node-id напрямую, ремап не нужен.
d = DS["data"]; src = d.src.numpy(); dst = d.dst.numpy(); C = DS["num_classes"]
assert int(dst.max()) < C, "items НЕ в [0,C) — idea 1 несовместима по индексам"
print(f"items (dst): [{dst.min()}, {dst.max()}]  uniq={len(np.unique(dst))}  (== num_classes={C})")
print(f"users (src): [{src.min()}, {src.max()}]  uniq={len(np.unique(src))}")
print(f"пересечение id юзеров/айтемов: {len(set(np.unique(src)) & set(np.unique(dst)))}  → M[u,:] ранжирует item-ноды напрямую")

items (dst): [0, 512]  uniq=513  (== num_classes=513)
users (src): [513, 1504]  uniq=992
пересечение id юзеров/айтемов: 0  → M[u,:] ранжирует item-ноды напрямую


### Коллектор `stream_sampler` — один строго-каузальный проход

В lockstep с тем же `pm = t < lab_ts0` split (как p4 `stream_diag`) ведём **три состояния, записываемые только прошлым**:
1. **Реальный** `LastNeighborLoader(num_nodes, size=K)` — `.insert(pm)` перед чтением, `.insert(~pm)` после ⇒ точный recency-набор (не аппроксимация);
2. `HawkesMemory` `M`;
3. `Cuv[N,C]` (support) + `cnt[N]` (warmth).

На каждый **тестовый** label-день, для каждого размеченного юзера с ≥1 позитивом, сравниваем **два набора-кандидата размера ≤K**:
- `rec` = `unique(nl.neighbors[u][nl.e_id[u]>=0])` — недавние item'ы; `E_rec` = число валидных слотов (= стоимость GNN);
- `m_topk` = top-K item'ов по `M.read(u)` (где `M>0`).

Скаляры на `(user, day)`: `jaccard(rec, m_topk)`; покрытие позитивов набором (`npos_*`, `posmass_*`); **within-set потолки** `rec_ceil=ndcg(yi, yi·rec_mask)`, `m_ceil=ndcg(yi, yi·m_mask)`, `support_ceil=ndcg(yi, yi·seen)`; паритет `m_read=ndcg(yi, M_row)`. Дельта `m_ceil − rec_ceil` изолирует **чистый эффект отбора** (одинаковая идеализация ранжирования сокращается). `M` только выбирает набор — никогда не предсказывает.

In [3]:
def topk_pos(row, K):  # top-K индексов среди ненулевых M по убыванию значения
    idx = np.nonzero(row > 0)[0]
    if idx.size > K:
        idx = idx[np.argpartition(-row[idx], K)[:K]]
    return idx

def stream_sampler(DS, mem, nl, cnt, Cuv, K, dl, collect=False):
    ds = DS["ds"]; label_t = ds.get_label_time(); C = DS["num_classes"]; recs = []
    def insert(s, dd):
        if s.size:
            nl.insert(torch.from_numpy(np.ascontiguousarray(s, dtype=np.int64)),
                      torch.from_numpy(np.ascontiguousarray(dd, dtype=np.int64)))
    for batch in dl:
        sn, dn, tn = batch.src.numpy(), batch.dst.numpy(), batch.t.numpy(); wn = batch.msg[:, 0].numpy()
        if float(batch.t[-1]) > label_t:
            lt = ds.get_node_label(batch.t[-1])
            if lt is None: break
            lab_ts0 = float(lt[0][0]); ls, labs = lt[1].numpy(), lt[2].numpy()
            label_t = ds.get_label_time(); pm = tn < lab_ts0
            mem.update(sn[pm], dn[pm], tn[pm], wn[pm]); np.add.at(cnt, sn[pm], 1)
            np.add.at(Cuv, (sn[pm].astype(np.int64), dn[pm].astype(np.int64)), 1); insert(sn[pm], dn[pm])
            if collect:
                Mday = mem.read(ls, lab_ts0)
                lt_idx = torch.from_numpy(ls.astype(np.int64))
                recN = nl.neighbors[lt_idx].numpy(); recE = nl.e_id[lt_idx].numpy()
                for i in range(len(ls)):
                    yi = labs[i]; tot = yi.sum()
                    if tot <= 0: continue
                    u = int(ls[i]); m_row = Mday[i]; valid = recE[i] >= 0
                    rec_items = np.unique(recN[i][valid]) if valid.any() else np.empty(0, np.int64)
                    m_idx = topk_pos(m_row, K)
                    rec_mask = np.zeros(C, bool); rec_mask[rec_items] = True
                    m_mask = np.zeros(C, bool); m_mask[m_idx] = True
                    pos = yi > 0; seen = Cuv[u] > 0
                    sr, sm = set(rec_items.tolist()), set(m_idx.tolist())
                    union = len(sr | sm); jac = (len(sr & sm) / union) if union else 1.0
                    Y = yi[None]
                    recs.append((u, int(cnt[u]), int(seen.sum()), int(pos.sum()),
                                 int((pos & rec_mask).sum()), int((pos & m_mask).sum()), int((pos & m_mask & ~rec_mask).sum()),
                                 float(yi[rec_mask].sum() / tot), float(yi[m_mask].sum() / tot),
                                 len(rec_items), int(valid.sum()), len(m_idx), jac,
                                 float(ndcg_score(Y, (yi * rec_mask)[None], k=10)),
                                 float(ndcg_score(Y, (yi * m_mask)[None], k=10)),
                                 float(ndcg_score(Y, (yi * seen)[None], k=10)),
                                 float(ndcg_score(Y, m_row[None], k=10)),
                                 int(np.all(Cuv[u, m_idx] > 0)) if len(m_idx) else 1))
            sn, dn, tn, wn = sn[~pm], dn[~pm], tn[~pm], wn[~pm]
        mem.update(sn, dn, tn, wn); np.add.at(cnt, sn, 1)
        np.add.at(Cuv, (sn.astype(np.int64), dn.astype(np.int64)), 1); insert(sn, dn)
    return recs

mem = HawkesMemory(DS["num_nodes"], C, alpha, phi)
nl = LastNeighborLoader(DS["num_nodes"], size=K)
cnt = np.zeros(DS["num_nodes"]); Cuv = np.zeros((DS["num_nodes"], C), dtype=np.int32)
stream_sampler(DS, mem, nl, cnt, Cuv, K, DS["loaders"]["train"])
stream_sampler(DS, mem, nl, cnt, Cuv, K, DS["loaders"]["val"])
rs = stream_sampler(DS, mem, nl, cnt, Cuv, K, DS["loaders"]["test"], collect=True)
DS["ds"].reset_label_time()
sd = pl.DataFrame(rs, schema=["user","warmth","seen_items","n_pos","npos_rec","npos_m","npos_m_only",
                              "posmass_rec","posmass_m","rec_distinct","E_rec","m_k","jaccard",
                              "rec_ceil","m_ceil","support_ceil","m_read","m_all_seen"], orient="row")
print(f"n={sd.height}  ПАРИТЕТ m_read={sd['m_read'].mean():.4f} (p4=0.5203)  "
      f"m_all_seen={sd['m_all_seen'].mean():.3f}  E_rec med={int(sd['E_rec'].median())}  m_k med={int(sd['m_k'].median())}")

n=32151  ПАРИТЕТ m_read=0.5203 (p4=0.5203)  m_all_seen=1.000  E_rec med=30  m_k med=30


In [4]:
# Premise A (overlap) + B (восстановление позитивов) + PROXY (потолки) — ключевые числа гейта
print("OVERALL:")
print(f"  Jaccard(recency,M): med={sd['jaccard'].median():.3f} mean={sd['jaccard'].mean():.3f}")
print(f"  потолки: rec_ceil={sd['rec_ceil'].mean():.4f}  m_ceil={sd['m_ceil'].mean():.4f}  "
      f"support_ceil={sd['support_ceil'].mean():.4f}  (m_read={sd['m_read'].mean():.4f})")
print(f"  ДЕЛЬТА m_ceil-rec_ceil = {sd['m_ceil'].mean()-sd['rec_ceil'].mean():+.4f}   "
      f"(чистый эффект отбора набора)")
print(f"  posmass: rec={sd['posmass_rec'].mean():.4f}  m={sd['posmass_m'].mean():.4f}  "
      f"восстановлено={sd['posmass_m'].mean()-sd['posmass_rec'].mean():+.4f}")
print(f"  доля (user,day) где M достаёт пропущенный recency позитив: {(sd['npos_m_only']>0).mean()*100:.1f}%")
print("\nПо warmth-когортам (холодные→тёплые):")
cb = (sd.with_columns(pl.col("warmth").qcut(5, allow_duplicates=True).alias("b"))
      .group_by("b").agg(
          pl.col("warmth").median().cast(pl.Int64).alias("med_w"),
          pl.col("jaccard").median().round(3).alias("jac_med"),
          pl.col("seen_items").median().cast(pl.Int64).alias("seen"),
          pl.col("rec_distinct").median().cast(pl.Int64).alias("rec_dist"),
          pl.col("rec_ceil").mean().round(4).alias("rec_ceil"),
          pl.col("m_ceil").mean().round(4).alias("m_ceil"),
          (pl.col("m_ceil") - pl.col("rec_ceil")).mean().round(4).alias("d_ceil"),
          (pl.col("npos_m_only") > 0).mean().round(3).alias("recov"),
          pl.len().alias("n")).sort("med_w"))
print(cb)

OVERALL:
  Jaccard(recency,M): med=0.237 mean=0.247
  потолки: rec_ceil=0.5512  m_ceil=0.8901  support_ceil=0.9867  (m_read=0.5203)
  ДЕЛЬТА m_ceil-rec_ceil = +0.3388   (чистый эффект отбора набора)
  posmass: rec=0.4445  m=0.8237  восстановлено=+0.3792
  доля (user,day) где M достаёт пропущенный recency позитив: 86.8%

По warmth-когортам (холодные→тёплые):
shape: (5, 10)
┌────────────────┬───────┬─────────┬──────┬───┬────────┬────────┬───────┬──────┐
│ b              ┆ med_w ┆ jac_med ┆ seen ┆ … ┆ m_ceil ┆ d_ceil ┆ recov ┆ n    │
│ ---            ┆ ---   ┆ ---     ┆ ---  ┆   ┆ ---    ┆ ---    ┆ ---   ┆ ---  │
│ cat            ┆ i64   ┆ f64     ┆ i64  ┆   ┆ f64    ┆ f64    ┆ f64   ┆ u32  │
╞════════════════╪═══════╪═════════╪══════╪═══╪════════╪════════╪═══════╪══════╡
│ (-inf, 6715]   ┆ 3259  ┆ 0.25    ┆ 98   ┆ … ┆ 0.8632 ┆ 0.3201 ┆ 0.827 ┆ 6431 │
│ (6715, 15388]  ┆ 10596 ┆ 0.242   ┆ 132  ┆ … ┆ 0.8769 ┆ 0.3506 ┆ 0.851 ┆ 6430 │
│ (15388, 26316] ┆ 20438 ┆ 0.235   ┆ 168  ┆ … ┆ 0.884  ┆ 0

In [5]:
# Визуализация: потолки набора по warmth + распределение overlap
plot = (sd.with_columns(pl.col("warmth").qcut(5, allow_duplicates=True).alias("b"))
        .group_by("b").agg(pl.col("warmth").median().cast(pl.Int64).alias("med_w"),
                           pl.col("rec_ceil").mean().alias("recency-30"),
                           pl.col("m_ceil").mean().alias("M-top30"),
                           pl.col("support_ceil").mean().alias("вся история (support)"))
        .sort("med_w").to_pandas())
fig = go.Figure()
for col, color in [("recency-30", "#d7191c"), ("M-top30", "#2c7fb8"), ("вся история (support)", "#999999")]:
    fig.add_bar(name=col, x=plot["med_w"].astype(str), y=plot[col],
                text=[f"{v:.2f}" for v in plot[col]], textposition="outside",
                marker_color=color, opacity=0.6 if "история" in col else 1.0)
fig.update_layout(barmode="group", height=440,
                  title="Within-set потолок NDCG@10 по warmth-когортам: M-набор ≫ recency (равный бюджет K=30)",
                  xaxis_title="медианная warmth когорты", yaxis_title="within-set ceil NDCG@10",
                  legend=dict(orientation="h", y=-0.2))
fig.show()

fig2 = px.histogram(sd.select("jaccard").to_pandas(), x="jaccard", nbins=40,
                    title=f"Распределение Jaccard(recency, M-top-K): медиана={sd['jaccard'].median():.3f} — наборы почти не пересекаются")
fig2.update_layout(height=340, xaxis_title="Jaccard overlap наборов соседей", yaxis_title="#(user,day)")
fig2.show()

In [6]:
# Feasibility & CPU-стоимость селекции
tt = float(d.t.numpy().max()); users = np.unique(d.src.numpy())[:200]
def msel():
    Mr = mem.read(users, tt)
    return [topk_pos(Mr[i], K) for i in range(len(users))]
t_m = timeit.timeit(msel, number=20) / 20 * 1000
mb = (mem.M.nbytes + Cuv.nbytes) / 1e6
print(f"M-селекция {len(users)} юзеров: {t_m:.2f} мс/день  →  ×~237 тест-дней/эпоху ≈ {t_m*237/1000:.2f} с/эпоху (пренебрежимо)")
print(f"буферы: M+Cuv = {mb:.1f} МБ (плотные [{DS['num_nodes']}×{C}])  |  m_all_seen={sd['m_all_seen'].mean():.3f} "
      f"(у каждого M-выбранного item есть прошлое событие → edge-фичи t/msg синтезируемы из last_eid-буфера в P5)")
print(f"бюджет GNN равный: E_rec med={int(sd['E_rec'].median())} vs E_M(=m_k) med={int(sd['m_k'].median())}")

M-селекция 200 юзеров: 1.00 мс/день  →  ×~237 тест-дней/эпоху ≈ 0.24 с/эпоху (пренебрежимо)
буферы: M+Cuv = 9.3 МБ (плотные [1505×513])  |  m_all_seen=1.000 (у каждого M-выбранного item есть прошлое событие → edge-фичи t/msg синтезируемы из last_eid-буфера в P5)
бюджет GNN равный: E_rec med=30 vs E_M(=m_k) med=30


## Вердикт IDEA 1 — **GO** ✅

Паритет `m_read=0.5203` (= p4) подтверждает каузальность пробы.

| гейт | порог | измерено | вердикт |
|---|---|---|---|
| A. наборы различны | Jaccard med < ~0.6 | **0.237** | ✅ сильно |
| B. M достаёт пропущенные позитивы | recovery > 0 | **86.8%** строк; +0.379 массы | ✅ сильно |
| proxy. M-вход лучше recency | m_ceil − rec_ceil > +0.01 | **+0.339** (0.890 vs 0.551), равномерно по когортам | ✅ сильно |
| C. покрытие | — | recency обрезан на K=30, M выбирает 30 лучших из 98–202 виденных | ✅ |
| стоимость / осуществимость | пренебрежима | 0.24 с/эпоху, 9.3 МБ, edge-фичи синтезируемы | ✅ |

**Механизм.** Recency кормит GNN последними 30 *событиями* (часто дубликаты), выбрасывая 70–170 различных item'ов истории. M-top-30 берёт 30 *самых аффинных* — его within-set потолок (0.89) почти дотягивает до потолка всей истории (0.987), recency (0.55) — нет. При равном бюджете соседей это строго лучший вход.

**Честная оговорка.** Потолки — проксиверхняя оценка: декодер выдаёт логиты по всем 513 классам, не ограничен набором. Дельта изолирует эффект *отбора* (идеализация ранжирования сокращается), но реальный прирост обучаемой модели решает только эксперимент.

**No-ensemble / каузальность** соблюдены: M только выбирает вход единственной головы; на инференсе второго предсказателя нет. M и loader пишутся строго прошлым (`t<lab_ts0`).

**→ Сидим P5-задачу:** A/B обучаемой TGNv2 с M-driven сэмплером соседей vs recency `LastNeighborLoader`, genre, 3 сида, метрика overall test NDCG@10 (+ срез по warmth). Seam: `neighbor_loader(...)` в `train-tgbn-nodeproppred.py` L132/L261; M обновляется в lockstep в `process_edges` (L180), edge-фичи top-K из `last_eid[N,C]`-буфера. **Фальсификатор:** Jaccard≈1 или дельта потолков ≈0 — здесь оба опровергнуты.

## Дизайн метода — разреженность `M` и сколько соседей нужно (ВСЕ датасеты)

Вопрос: сэмплируем K соседей, но у юзера лишь `m < K` ненулевых `M` — остальные K−m = шум нулевой аффинности. Две вещи измеряем строго-каузально на genre/reddit/token:
1. **Как часто `seen_items < K`** (когда padding вообще возникает) — по датасетам сильно разнится (на token холодных много).
2. **Кривая покрытия `posmass@K`** = доля позитивной массы дня в top-K item'ах по `M` (вариативный размер: cap на `seen`, нули не добавляем). Где насыщается → сколько соседей нужно; ранний плато → хвост по рангу `M` это «шум», кандидат на порог.

Зазор `1 − posmass_support` = позитивы, которых нет в seen вообще (недостижимы никаким отбором соседей — это coverage_gap из p4). _(trade симметричен, `items≈users` — рычаг неприменим, пропускаем.)_

In [8]:
# Кросс-датасетный скан: seen_items + кривая покрытия posmass@K (дёшево, без ndcg/loader)
def coverage_scan(name, Kcuts=(1, 3, 5, 10, 20, 30, 50, 100)):
    Dd = DS if name == "tgbn-genre" else load_dataset(name)
    ds = Dd["ds"]; N, Cc = Dd["num_nodes"], Dd["num_classes"]
    ph = make_rank_phi(Dd); a = np.log(2) / (KAPPA * 86400.0)
    mem = HawkesMemory(N, Cc, a, ph); cnt = np.zeros(N); recs = []; Kmax = max(Kcuts)
    def stream(dl, collect):
        label_t = ds.get_label_time()
        for batch in dl:
            sn, dn, tn = batch.src.numpy(), batch.dst.numpy(), batch.t.numpy(); wn = batch.msg[:, 0].numpy()
            if float(batch.t[-1]) > label_t:
                lt = ds.get_node_label(batch.t[-1])
                if lt is None: break
                lab_ts0 = float(lt[0][0]); ls, labs = lt[1].numpy(), lt[2].numpy()
                label_t = ds.get_label_time(); pm = tn < lab_ts0
                mem.update(sn[pm], dn[pm], tn[pm], wn[pm]); np.add.at(cnt, sn[pm], 1)
                if collect:
                    Md = mem.read(ls, lab_ts0)
                    for i in range(len(ls)):
                        yi = labs[i]; tot = yi.sum()
                        if tot <= 0: continue
                        mr = Md[i]; seen = int((mr > 0).sum())
                        order = np.argsort(-mr)[:Kmax]; cum = np.cumsum(yi[order])
                        pms = [float(cum[min(k, seen) - 1] / tot) if seen > 0 else 0.0 for k in Kcuts]
                        recs.append((int(cnt[ls[i]]), seen, int((yi > 0).sum()),
                                     float(yi[mr > 0].sum() / tot), *pms))
                sn, dn, tn, wn = sn[~pm], dn[~pm], tn[~pm], wn[~pm]
            mem.update(sn, dn, tn, wn); np.add.at(cnt, sn, 1)
    stream(Dd["loaders"]["train"], False); stream(Dd["loaders"]["val"], False)
    stream(Dd["loaders"]["test"], True); ds.reset_label_time()
    cols = ["warmth", "seen", "n_pos", "posmass_support"] + [f"pm@{k}" for k in Kcuts]
    return pl.DataFrame(recs, schema=cols, orient="row")

def summ(name, cv, K=30):
    print(f"\n=== {name} ===  n={cv.height}  seen: med={int(cv['seen'].median())} "
          f"p10={int(cv['seen'].quantile(0.1))}  | seen<{K}: {(cv['seen']<K).mean()*100:.1f}%  "
          f"seen<10: {(cv['seen']<10).mean()*100:.1f}%")
    print(f"  posmass@K:  @1={cv['pm@1'].mean():.3f}  @5={cv['pm@5'].mean():.3f}  @10={cv['pm@10'].mean():.3f}  "
          f"@30={cv['pm@30'].mean():.3f}  @100={cv['pm@100'].mean():.3f}  support(all seen)={cv['posmass_support'].mean():.3f}")

cov = {}
for nm in ["tgbn-genre", "tgbn-reddit"]:
    cov[nm] = coverage_scan(nm); summ(nm, cov[nm])


=== tgbn-genre ===  n=32151  seen: med=154 p10=83  | seen<30: 0.9%  seen<10: 0.2%
  posmass@K:  @1=0.147  @5=0.429  @10=0.593  @30=0.824  @100=0.959  support(all seen)=0.980


0it [00:00, ?it/s]

90997it [00:00, 909806.01it/s]

189369it [00:00, 953254.06it/s]

298794it [00:00, 1017617.10it/s]

...

11862337it [00:12, 943430.27it/s]

11960117it [00:12, 953530.68it/s]

12055580it [00:12, 953849.55it/s]

12077151it [00:12, 987076.07it/s]


=== tgbn-reddit ===  n=517845  seen: med=40 p10=22  | seen<30: 25.0%  seen<10: 0.5%
  posmass@K:  @1=0.272  @5=0.601  @10=0.762  @30=0.929  @100=0.966  support(all seen)=0.966


In [9]:
cov["tgbn-token"] = coverage_scan("tgbn-token"); summ("tgbn-token", cov["tgbn-token"])

0it [00:00, ?it/s]

59304it [00:00, 592934.54it/s]

123046it [00:00, 619093.08it/s]

184956it [00:00, 556791.94it/s]

241302it [00:00, 559020.05it/s]

...

5278648it [00:09, 546407.89it/s]

5333974it [00:09, 546931.95it/s]

5388755it [00:10, 538352.56it/s]

5397704it [00:10, 535691.50it/s]


=== tgbn-token ===  n=250550  seen: med=25 p10=1  | seen<30: 54.9%  seen<10: 31.4%
  posmass@K:  @1=0.345  @5=0.489  @10=0.543  @30=0.603  @100=0.623  support(all seen)=0.623


In [10]:
# Кросс-датасетные кривые покрытия + частота padding (seen<K)
Kcuts = [1, 3, 5, 10, 20, 30, 50, 100]
long = []
for nm, cv in cov.items():
    sh = nm.replace("tgbn-", "")
    for k in Kcuts:
        long.append((sh, k, cv[f"pm@{k}"].mean()))
    long.append((sh, 160, cv["posmass_support"].mean()))   # асимптота = вся история (support)
lf = pl.DataFrame(long, schema=["датасет", "K", "posmass"], orient="row").to_pandas()
fig = px.line(lf, x="K", y="posmass", color="датасет", markers=True, log_x=True,
              title="Покрытие позитивной массы дня top-K соседями по M (K=160 ≈ вся история/support)",
              labels={"posmass": "доля позитивной массы метки", "K": "K соседей (log)"})
fig.update_layout(height=430)
fig.show()

pad = pl.DataFrame({"датасет": ["genre", "reddit", "token"],
                    "seen<10": [(cov[f"tgbn-{n}"]["seen"] < 10).mean() for n in ["genre", "reddit", "token"]],
                    "seen<30": [(cov[f"tgbn-{n}"]["seen"] < 30).mean() for n in ["genre", "reddit", "token"]]})
fig2 = px.bar(pad.to_pandas().melt(id_vars="датасет", var_name="порог", value_name="доля"),
              x="датасет", y="доля", color="порог", barmode="group", text_auto=".1%",
              title="Как часто seen_items < K (когда padding-шум реален): genre≈0, token>половины")
fig2.update_layout(height=360, yaxis_tickformat=".0%")
fig2.show()
print("posmass_support (потолок достижимого отбором соседей):",
      {n: round(cov[f"tgbn-{n}"]["posmass_support"].mean(), 3) for n in ["genre", "reddit", "token"]})

posmass_support (потолок достижимого отбором соседей): {'genre': 0.98, 'reddit': 0.966, 'token': 0.623}


### Выводы по дизайну (кросс-датасетно)

1. **Вариативный размер набора (только `M>0`, без нулевого padding) — обязателен, и важность растёт с холодностью.** Частота `seen<30`: genre **0.9%** → reddit **25%** → token **55%** (31% с `seen<10`). Твоя интуиция верна; **на genre проблема почти не видна — поэтому «по всем датасетам» и было правильным**. На token больше половины предсказаний иначе кормились бы шумом.

2. **K — это ВЕРХНЯЯ ГРАНИЦА, а не фикс. число.** Брать `min(#seen, K_cap)`. Кривые покрытия разные: genre растёт до ~100 (`@30`=0.82, `@100`=0.96 — хочет больше соседей), reddit насыщается к ~30, token к ~10 (и так почти у всех `seen<30`). Щедрый `K_cap` + вариативный размер авто-адаптируется под датасет.

3. **Хвост по рангу `M` — НЕ шум (на genre/reddit), он информативен.** Кривая genre монотонно растёт до 100 ⇒ каждый добавленный сосед несёт позитивную массу. Значит абсолютный порог «отрезать низкий `M`» НЕ оправдан — единственный шум это **нулевая** аффинность (padding). Различать: `M=0` (unseen) → выбросить; малый, но `>0` `M` → оставить.

4. **У token жёсткий потолок достижимости `support=0.623`** — 38% позитивов вообще не в seen ⇒ сэмплер один упирается в 0.62 (это coverage_gap из p4; чинится только item-side достройкой — предел идеи 1 на token).

## Меню модификаций метода (design-workflow) + стратегический пересмотр

**Ответ на «шумовые соседи»:** валидно по сути, но **уже решено** дизайном. Сэмплер не паддит нулями: `topk_pos`/реальный loader (`mask=e_id>=0`, mtgn.py:271-273) отдают только валидные рёбра ДО `TransformerConv`; холодный юзер с 0 соседей деградирует на свою память (`root_weight=True`). На genre сценарий редок (`seen<30`≈1%), реален на reddit (25%) / token (55%). ⇒ **зафиксировать вариативный размер как явный контракт** (бесплатно, правильно), но не зацикливаться — главный рычаг в другом.

**⚠️ Стратегический пересмотр (важно):** M-only read уже = **0.5203 (паритет с обученной TGNv2)**. Значит планка успеха — **бить 0.520, а не 0.469**. Вопрос не «лучше ли M-набор» (доказано: да), а «**добавляет ли GNN что-то поверх ранжирования M**». Все варианты обязаны A/B-иться против `use_gnn=False`/M-only.

**Ранжированное меню (single-model, M строго ДО единственной головы):**
| # | вариант | суть | где живёт |
|---|---|---|---|
| **V1** | M-сэмплер с **развязанным read-budget `K_M`** | отдавать `min(#seen, K_M)` по `M>0`; `K_M` независим от recency-size (genre хочет K_M≫30 — кривая не насыщена) | форк `LastNeighborLoader.__call__`, swap L132/261 |
| **V2** | **M как per-edge attention-фича** | `m_e` в `edge_attr` → TransformerConv взвешивает соседей по аффинности (не только видит набор). *Центральная новизна.* | embmodule L22/28, gnn() L137/266 |
| **V3** | **selection half-life `κ_select`** | один скаляр-ключ ранжирования; M уже = recency-decayed-affinity ⇒ subsumes recency-гибрид | в сэмплере |
| V4 | M как аддитивный bias в attention (`m_coef`) | интерпретируемый скаляр: помогает ли M и насколько | subclass `message()` |
| V5 | трим низкого `M` (`τ`) vs мягкое взвешивание | хвост Хокса `M>0` навсегда после 1 касания → старые рёбра/конфузеры | поверх V1 / =V2 |
| V6 | **сводка** строки M (support/entropy/Gini) как node-фича | conditioning головы на «уверенность» юзера | decoder.py in_dim |

**Жёсткая красная линия (ансамбли):** запрещено `logit += α·log M` на выходе; запрещена **полная выровненная строка** `M[u,:]` как node-фича (голова выучит `GNN_logits + linear(M)` = скрытый аддитивный ансамбль). Разрешены: M как edge-фича/bias, **перестановочно-инвариантные** сводки строки.

**Дешёвые офлайн-гейты (до обучения):** **E1** — логрегрессия `P(i позитивен завтра | rel_time, M[u,i])`: даёт ли M прирост AUC над recency ⇒ жив ли V2/V4. E_K — m_ceil(K) на сетке K (где колено `K_M`). E_κ — свип `κ_select`. E_stale — распределение давности M-выбранных рёбер.